# 中芯国际（688981.SH）交易数据获取与可视化

本 Notebook 演示了通过 **Tushare API** 获取中芯国际近一年日线行情数据，并使用 **Matplotlib** 绘制 K线图和交易量图的完整流程。

## 流程概述
1. 调用 Tushare HTTP API 获取日线行情数据
2. 数据清洗与格式转换
3. 存储为本地 JSON / CSV 文件
4. 使用 Matplotlib 绘制 K线图（含均线）
5. 绘制交易量柱状图
6. 月度统计分析

## 数据来源
- **接口**: Tushare `daily`（日线行情）
- **标的**: 中芯国际 688981.SH（科创板）
- **时间范围**: 2025-07-04 ~ 2026-07-03

## 1. 导入依赖库

In [ ]:
import urllib.request
import json
import csv
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import Rectangle
from datetime import datetime, timedelta

# 设置中文字体（macOS）
plt.rcParams["font.sans-serif"] = ["Arial Unicode MS", "Heiti TC", "PingFang SC"]
plt.rcParams["axes.unicode_minus"] = False

print("库导入完成 ✅")

## 2. 配置 Tushare API

设置 API Token 和请求参数。Tushare 采用 POST 方式调用，传入 `api_name`、`token`、`params` 和 `fields`。

In [ ]:
# Tushare API 配置
TOKEN = "your_tushare_token_here"  # 替换为你的 Tushare Token
API_URL = "https://api.tushare.pro"

# 查询参数
TS_CODE = "688981.SH"       # 中芯国际（科创板）
START_DATE = "20250704"    # 近一年起始日期
END_DATE = "20260704"      # 近一年结束日期

print(f"标的: {TS_CODE} (中芯国际)")
print(f"时间范围: {START_DATE} ~ {END_DATE}")

### 2.1 定义 API 调用函数

封装 Tushare HTTP API 调用逻辑。

In [ ]:
def call_tushare(api_name, params, fields=None):
    """调用 Tushare HTTP API"""
    payload = {"api_name": api_name, "token": TOKEN, "params": params}
    if fields:
        payload["fields"] = fields
    data = json.dumps(payload).encode("utf-8")
    req = urllib.request.Request(API_URL, data=data, headers={"Content-Type": "application/json"})
    resp = urllib.request.urlopen(req, timeout=60)
    return json.loads(resp.read().decode("utf-8"))

print("API 调用函数已定义 ✅")

## 3. 获取日线行情数据

调用 `daily` 接口获取中芯国际的 OHLCV 数据。

**字段说明：**
| 字段 | 含义 |
|---|---|
| `open` | 开盘价 |
| `high` | 最高价 |
| `low` | 最低价 |
| `close` | 收盘价 |
| `pre_close` | 前收盘价 |
| `change` | 涨跌额 |
| `pct_chg` | 涨跌幅（%）|
| `vol` | 成交量（手）|
| `amount` | 成交额（千元）|

In [ ]:
# 调用 daily 接口获取日线数据
response = call_tushare(
    api_name="daily",
    params={"ts_code": TS_CODE, "start_date": START_DATE, "end_date": END_DATE},
    fields="ts_code,trade_date,open,high,low,close,pre_close,change,pct_chg,vol,amount"
)

if response.get("code") == 0 and response.get("data", {}).get("items"):
    data = response["data"]
    fields_list = data["fields"]
    items = data["items"]
    print(f"数据获取成功！共 {len(items)} 条记录")
else:
    print(f"获取失败: {response.get('msg', 'unknown error')}")

## 4. 数据清洗与转换

将原始数据转为 Pandas DataFrame，进行类型转换和排序。

In [ ]:
# 转为 DataFrame
df = pd.DataFrame(items, columns=fields_list)

# 类型转换
numeric_cols = ["open", "high", "low", "close", "pre_close", "change", "pct_chg", "vol", "amount"]
df[numeric_cols] = df[numeric_cols].astype(float)

# 日期格式转换
df["trade_date"] = pd.to_datetime(df["trade_date"], format="%Y%m%d")
df.set_index("trade_date", inplace=True)
df.sort_index(inplace=True)

print(f"数据范围: {df.index[0].strftime('%Y-%m-%d')} ~ {df.index[-1].strftime('%Y-%m-%d')}")
print(f"交易日数: {len(df)}")
df.head(10)

### 4.1 数据概览统计

In [ ]:
print("=== 价格统计 ===")
print(f"最高价: {df["high"].max():.2f}  ({df["high"].idxmax().strftime("%Y-%m-%d")})")
print(f"最低价: {df["low"].min():.2f}  ({df["low"].idxmin().strftime("%Y-%m-%d")})")
print(f"期间涨跌幅: {((df["close"].iloc[-1] / df["close"].iloc[0]) - 1) * 100:.2f}%")
print(f"日均成交量: {df["vol"].mean():,.0f} 手")
print()
df.describe()

## 5. 存储数据到本地

将数据保存为 JSON 和 CSV 两种格式。

In [ ]:
# 存储为 JSON
json_path = "smic_daily.json"
records = df.reset_index().to_dict(orient="records")
for r in records:
    r["trade_date"] = r["trade_date"].strftime("%Y%m%d")
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=2)
print(f"JSON 已保存: {json_path} ({len(records)} 条)")

# 存储为 CSV
csv_path = "smic_daily.csv"
df.to_csv(csv_path, encoding="utf-8-sig")
print(f"CSV 已保存: {csv_path}")

## 6. 计算技术指标

计算移动平均线（MA5、MA20、MA60）。

In [ ]:
# 计算移动平均线
df["MA5"] = df["close"].rolling(window=5).mean()
df["MA20"] = df["close"].rolling(window=20).mean()
df["MA60"] = df["close"].rolling(window=60).mean()

print("均线计算完成 ✅")
df[["close", "MA5", "MA20", "MA60"]].tail(10)

## 7. 绘制 K 线图

使用 Matplotlib 手动绘制蜡烛图，叠加均线。

> **A股配色**：涨为红色，跌为绿色

In [ ]:
def draw_candlestick(ax, df_data):
    """绘制 K 线蜡烛图"""
    width = 0.6
    for i in range(len(df_data)):
        x = mdates.date2num(df_data.index[i])
        o, h, l, c = df_data["open"].iloc[i], df_data["high"].iloc[i], df_data["low"].iloc[i], df_data["close"].iloc[i]
        
        color = "#ef4444" if c >= o else "#22c55e"  # 涨红跌绿
        
        # 影线
        ax.plot([x, x], [l, h], color=color, linewidth=0.8, zorder=2)
        
        # 实体
        bottom, height = min(o, c), abs(c - o)
        if height == 0: height = 0.01
        rect = Rectangle((x - width/2, bottom), width, height, facecolor=color, edgecolor=color, linewidth=0.8, zorder=3)
        ax.add_patch(rect)

print("K线绘制函数已定义 ✅")

In [ ]:
# 创建 K 线图
fig, ax = plt.subplots(figsize=(16, 8))
draw_candlestick(ax, df)

# 叠加均线
ax.plot(mdates.date2num(df.index), df["MA5"], label="MA5", color="#f59e0b", linewidth=1, alpha=0.8)
ax.plot(mdates.date2num(df.index), df["MA20"], label="MA20", color="#3b82f6", linewidth=1, alpha=0.8)
ax.plot(mdates.date2num(df.index), df["MA60"], label="MA60", color="#8b5cf6", linewidth=1, alpha=0.8)

ax.set_title("中芯国际 (688981.SH) 日K线图  2025.07 ~ 2026.07", fontsize=16, fontweight="bold")
ax.set_xlabel("日期", fontsize=12)
ax.set_ylabel("价格（元）", fontsize=12)
ax.legend(loc="upper left", fontsize=11)
ax.grid(True, alpha=0.3, linestyle="--")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
ax.xaxis.set_major_locator(mdates.MonthLocator())
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("smic_kline.png", dpi=150, bbox_inches="tight")
plt.show()
print("K线图已保存 ✅")

## 8. 绘制交易量图

绘制与 K 线对应的成交量柱状图。

In [ ]:
# 创建交易量图
fig, ax = plt.subplots(figsize=(16, 4))

# 涨跌颜色
colors = ["#ef4444" if df["close"].iloc[i] >= df["open"].iloc[i] else "#22c55e" for i in range(len(df))]

ax.bar(mdates.date2num(df.index), df["vol"], color=colors, width=0.6, alpha=0.8)
ax.set_title("中芯国际 (688981.SH) 日成交量  2025.07 ~ 2026.07", fontsize=14, fontweight="bold")
ax.set_xlabel("日期", fontsize=12)
ax.set_ylabel("成交量（手）", fontsize=12)
ax.grid(True, alpha=0.3, linestyle="--", axis="y")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
ax.xaxis.set_major_locator(mdates.MonthLocator())
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("smic_volume.png", dpi=150, bbox_inches="tight")
plt.show()
print("交易量图已保存 ✅")

## 9. K线 + 交易量 组合图

上下排列，共享 X 轴。

In [ ]:
# 组合图
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 10), gridspec_kw={"height_ratios": [3, 1]}, sharex=True)

# 上图：K线
draw_candlestick(ax1, df)
ax1.plot(mdates.date2num(df.index), df["MA5"], label="MA5", color="#f59e0b", linewidth=1, alpha=0.8)
ax1.plot(mdates.date2num(df.index), df["MA20"], label="MA20", color="#3b82f6", linewidth=1, alpha=0.8)
ax1.plot(mdates.date2num(df.index), df["MA60"], label="MA60", color="#8b5cf6", linewidth=1, alpha=0.8)
ax1.set_title("中芯国际 (688981.SH)  K线 + 成交量  2025.07 ~ 2026.07", fontsize=16, fontweight="bold")
ax1.set_ylabel("价格（元）", fontsize=12)
ax1.legend(loc="upper left", fontsize=11)
ax1.grid(True, alpha=0.3, linestyle="--")

# 下图：成交量
ax2.bar(mdates.date2num(df.index), df["vol"], color=colors, width=0.6, alpha=0.8)
ax2.set_xlabel("日期", fontsize=12)
ax2.set_ylabel("成交量（手）", fontsize=12)
ax2.grid(True, alpha=0.3, linestyle="--", axis="y")

ax2.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
ax2.xaxis.set_major_locator(mdates.MonthLocator())
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("smic_kline_volume.png", dpi=150, bbox_inches="tight")
plt.show()
print("组合图已保存 ✅")

## 10. 收盘价走势图

In [ ]:
fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(mdates.date2num(df.index), df["close"], color="#2563eb", linewidth=1.5, label="收盘价")
ax.fill_between(mdates.date2num(df.index), df["close"], alpha=0.1, color="#2563eb")

# 标注最高最低点
high_idx = df["high"].idxmax()
low_idx = df["low"].idxmin()
ax.annotate(f"最高: {df.loc[high_idx, 'high']:.2f}", xy=(mdates.date2num(high_idx), df.loc[high_idx, "high"]),
            xytext=(10, 15), textcoords="offset points", arrowprops=dict(arrowstyle="->", color="red"), fontsize=11, color="red")
ax.annotate(f"最低: {df.loc[low_idx, 'low']:.2f}", xy=(mdates.date2num(low_idx), df.loc[low_idx, "low"]),
            xytext=(10, -25), textcoords="offset points", arrowprops=dict(arrowstyle="->", color="green"), fontsize=11, color="green")

ax.set_title("中芯国际 (688981.SH) 收盘价走势  2025.07 ~ 2026.07", fontsize=16, fontweight="bold")
ax.set_xlabel("日期", fontsize=12)
ax.set_ylabel("价格（元）", fontsize=12)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, linestyle="--")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
ax.xaxis.set_major_locator(mdates.MonthLocator())
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("smic_close_trend.png", dpi=150, bbox_inches="tight")
plt.show()
print("收盘价走势图已保存 ✅")

## 11. 月度收益统计

按月汇总涨跌幅和成交量。

In [ ]:
# 按月统计
df_m = df.copy()
df_m["year_month"] = df_m.index.to_period("M")

monthly = df_m.groupby("year_month").agg({"close": ["first", "last"], "vol": "sum"}).round(2)
monthly.columns = ["月初价", "月末价", "月成交量"]
monthly["月涨跌幅%"] = ((monthly["月末价"] / monthly["月初价"]) - 1) * 100
monthly["月涨跌幅%"] = monthly["月涨跌幅%"].round(2)
monthly

In [ ]:
# 月度涨跌幅柱状图
fig, ax = plt.subplots(figsize=(12, 5))
months = [str(m) for m in monthly.index]
returns = monthly["月涨跌幅%"].values
colors_m = ["#ef4444" if r >= 0 else "#22c55e" for r in returns]

bars = ax.bar(months, returns, color=colors_m, alpha=0.8, edgecolor="white", linewidth=0.5)
for bar, val in zip(bars, returns):
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + (0.5 if h > 0 else -1.5), f"{val:+.1f}%", ha="center", va="bottom" if h > 0 else "top", fontsize=10)

ax.set_title("中芯国际 月度涨跌幅统计", fontsize=14, fontweight="bold")
ax.set_xlabel("月份", fontsize=12)
ax.set_ylabel("涨跌幅（%）", fontsize=12)
ax.axhline(y=0, color="gray", linewidth=0.8)
ax.grid(True, alpha=0.3, linestyle="--", axis="y")
plt.tight_layout()
plt.savefig("smic_monthly_returns.png", dpi=150, bbox_inches="tight")
plt.show()
print("月度涨跌幅图已保存 ✅")

## 总结

本 Notebook 完成了以下工作：

| 步骤 | 内容 | 输出文件 |
|---|---|---|
| 1 | 通过 Tushare API 获取日线数据 | — |
| 2 | 数据清洗与 DataFrame 转换 | — |
| 3 | 存储为 JSON / CSV | `smic_daily.json`, `smic_daily.csv` |
| 4 | 计算 MA5/MA20/MA60 均线 | — |
| 5 | 绘制 K 线蜡烛图 | `smic_kline.png` |
| 6 | 绘制成交量柱状图 | `smic_volume.png` |
| 7 | K线 + 成交量组合图 | `smic_kline_volume.png` |
| 8 | 收盘价走势图 | `smic_close_trend.png` |
| 9 | 月度涨跌幅统计与可视化 | `smic_monthly_returns.png` |

> **注意**: 运行前需安装 `pandas`、`matplotlib`，并替换 `TOKEN` 为你的 Tushare Token。
>
> ```bash
> pip install pandas matplotlib
> ```